In [1]:
# ==============================================================================
# NOTEBOOK 4: ANALYSE THÉMATIQUE ET ANALYSE DE SENTIMENT PROFONDE
# PROJET: Text Mining - L'Enquêteur du Niger
# ==============================================================================

import pandas as pd
import numpy as np
import os
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from textblob import Blobber
from textblob_fr import PatternTagger, PatternAnalyzer

# 1. Configuration des chemins
INPUT_PATH = "../data/processed/enqueteur_preprocessed.csv"
FINAL_OUTPUT_PATH = "../data/final/enqueteur_final_analysis.csv"
FIGURES_DIR = "../outputs/figures/"

df = pd.read_csv(INPUT_PATH)
df['texte_tokenise'] = df['texte_tokenise'].fillna("")
print(f"Dataset chargé : {df.shape[0]} articles prêts pour l'analyse sémantique.")


Dataset chargé : 339 articles prêts pour l'analyse sémantique.


In [2]:
# ==============================================================================
# ÉTAPE 1 : ANALYSE DE SENTIMENT (Polarité et Subjectivité)
# ==============================================================================
print("\n--- Calcul des scores de sentiments avec TextBlob (FR) ---")

# Initialisation de TextBlob avec le module français
tb = Blobber(pos_tagger=PatternTagger(), analyzer=PatternAnalyzer())

def analyser_sentiments(text):
    if not text.strip():
        return 0.0, 0.0
    blob = tb(text)
    # sentiment[0] = polarité (de -1 très négatif à +1 très positif)
    # sentiment[1] = subjectivité (de 0 factuel à 1 très subjectif/engagé)
    return blob.sentiment[0], blob.sentiment[1]

# Application sur le texte nettoyé
sentiments = df['texte_tokenise'].apply(analyser_sentiments)
df['polarite'] = [s[0] for s in sentiments]
df['subjectivite'] = [s[1] for s in sentiments]

# Analyse rapide des scores obtenus
print(f"Polarité moyenne : {df['polarite'].mean():.3f} (Indique le ton global)")
print(f"Subjectivité moyenne : {df['subjectivite'].mean():.3f} (Indique le niveau d'engagement opinion/faits)")



--- Calcul des scores de sentiments avec TextBlob (FR) ---
Polarité moyenne : 0.065 (Indique le ton global)
Subjectivité moyenne : 0.221 (Indique le niveau d'engagement opinion/faits)


In [3]:
# ==============================================================================
# ÉTAPE 2 : TOPIC MODELING (LDA) - Extraction des thèmes de combat
# ==============================================================================
print("\n--- Extraction des grands thèmes de combat (LDA) ---")

# Nombre de thèmes à extraire (réglé sur 4 pour couvrir les grands axes du journal)
N_TOPICS = 4

# Vectorisation TF-IDF
tfidf_vectorizer = TfidfVectorizer(max_df=0.90, min_df=3, stop_words=None) # Stop words déjà traités
tfidf_matrix = tfidf_vectorizer.fit_transform(df['texte_tokenise'])

# Modèle LDA
lda_model = LatentDirichletAllocation(n_components=N_TOPICS, random_state=42, max_iter=15)
lda_output = lda_model.fit_transform(tfidf_matrix)

# Extraction des mots-clés par thème
words = tfidf_vectorizer.get_feature_names_out()
for topic_idx, topic in enumerate(lda_model.components_):
    top_words = [words[i] for i in topic.argsort()[:-10:-1]]
    print(f"Thème #{topic_idx + 1} : {', '.join(top_words)}")

# Assigner le thème dominant à chaque article
df['theme_dominant'] = np.argmax(lda_output, axis=1) + 1



--- Extraction des grands thèmes de combat (LDA) ---
Thème #1 : score, maroc, exact, faveur, stade, messages, mises, mœurs, secoue
Thème #2 : ne, mais, cette, niger, état, refondation, être, sans, 2025
Thème #3 : score, maroc, exact, faveur, stade, messages, mises, mœurs, secoue
Thème #4 : score, maroc, exact, faveur, stade, messages, mises, mœurs, secoue


In [4]:
# ==============================================================================
# ÉTAPE 3 : CROISEMENT THÈMES, SENTIMENTS ET ENGAGEMENT
# ==============================================================================
print("\n--- Synthèse par Thème dominant ---")
synthese = df.groupby('theme_dominant').agg({
    'post_id': 'count',
    'polarite': 'mean',
    'subjectivite': 'mean',
    'nb_partages': 'mean',
    'like': 'mean'
}).rename(columns={'post_id': 'nombre_articles'})

print(synthese.round(3))



--- Synthèse par Thème dominant ---
                nombre_articles  polarite  subjectivite  nb_partages    like
theme_dominant                                                              
1                             1     0.000         0.000         0.00   75.00
2                           338     0.065         0.221        20.03  210.32


In [5]:
# Sauvegarde finale
os.makedirs(os.path.dirname(FINAL_OUTPUT_PATH), exist_ok=True)
df.to_csv(FINAL_OUTPUT_PATH, index=False, encoding='utf-8')
print(f"\nAnalyses terminées ! Fichier final enregistré dans : {FINAL_OUTPUT_PATH}")


Analyses terminées ! Fichier final enregistré dans : ../data/final/enqueteur_final_analysis.csv
